In [1]:
import qiskit_metal as metal
from qiskit_metal import designs, draw, MetalGUI, Dict
import numpy as np
import pint
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', 'no-latex', 'notebook'])
from qiskit_metal.qlibrary.qubits.circle_transmon_squid import CircTransmonSQUID

design = designs.DesignPlanar()
design.renderers.qt.gds_renderer.options.plot_col_short = True
gui = MetalGUI(design)

In [2]:
# Global CPW trace geometry: applies to every RouteMeander (and any other
# QRoute-based component) that doesn't explicitly override trace_width/trace_gap.
design.variables['cpw_width'] = '20um'
design.variables['cpw_gap'] = '12.25um'

In [3]:
gui.canvas.draw_bbox = True

In [4]:
x_size = 5 #mm
y_size = 5 #mm

design.overwrite_enabled = True
design.chips.main.size.size_x = f'{x_size}mm'
design.chips.main.size.size_y = f'{y_size}mm'

In [5]:
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround

from design_helper import DesignHelper
from chip import Chip
from components import Resonator, Qubit

# inicializamos el helper
k_inductance_ratio = 0.00
helper = DesignHelper(alpha_inductance = k_inductance_ratio)
film_thickness = helper.ureg.Quantity(100, "nm")

In [6]:
from qiskit_metal import Dict

options = Dict(
    pos_x='0um', pos_y='0um', orientation='0', chip='main',
    cpw_width='cpw_width', cpw_gap='cpw_gap',

    pad_radius='90um',
    pad_gap='10um',

    jj_options=Dict(
        jj_width='5um',
        jj_angle='0',      # single JJ, straight down, away from the SQUID
        L_j='10nH',
        C_j='2fF',
        export_mask=False,
        jj_sim_gap='3um',
    ),

    squid_options=Dict(
        theta_2='180',        # SQUID sits at the top of the pad
        delta_angle='7.5',    # wide enough to see the two prongs clearly
        g1='4um',

        l1='300um',
        w1='6um',
        l2='300um',

        jj_a_options=Dict(
            jj_width='4um', L_j='15nH', C_j='1.5fF',
            export_mask=False, jj_sim_gap='3um',
        ),
        jj_b_options=Dict(
            jj_width='6um', L_j='8nH', C_j='1.8fF',
            export_mask=False, jj_sim_gap='3um',
        ),
    )
)

try:
    qubit_1.delete()
except NameError: 
    pass

# Pass ONLY the single options dictionary
qubit_1 = CircTransmonSQUID(design, 'Q1_SQUID', options=options)

gui.rebuild()
gui.autoscale()

## Readout resonators

In [124]:
# Creamos las terminaciones a ground necesarias cerca del main
try:
    launch_point1.delete()
    launch_point2.delete()
except NameError : pass


launch_point1 = OpenToGround(design, "launch_point1", options= dict(
    pos_x = "-350um",
    pos_y = "-850um",
    orientation = "0",
    termination_gap = "cpw_gap",
    gap = 'cpw_gap',
    width = 'cpw_width'
)
)
launch_point2 = ShortToGround(design, "launch_point2", options= dict(
    pos_x = "-150um",
    pos_y = "50um",
    orientation = "0",
    termination_gap = "cpw_gap",
    gap = 'cpw_gap',
    width = 'cpw_width'
)
)
gui.rebuild()
gui.autoscale()

In [131]:
from collections import OrderedDict
jogs = OrderedDict()
jogs[0] = ["L", "100um"]
jogs[1] = ["L", "565um"]
# jogs[2] = ["R", "200um"]
# jogs[3] = ["R", "560um"]
# jogs[4] = ["L", "200um"]
# jogs[5] = ["L", "600um"]
# jogs[6] = ["R", "200um"]
# jogs[7] = ["R", "600um"]
# jogs[8] = ["L", "200um"]
# jogs[1] = []
# jogs[1] = ["L", "500um"]
# jogs[2] = ["R", "200um"]
# jogs[3] = ["R", "500um"]

meander_1_options = dict(
    pin_inputs = dict(
        start_pin = dict(
            component = "launch_point1",
            pin = "open"
        ),
        end_pin = dict(
            component = "launch_point2",
            pin = "short"
        )
    ),
    fillet = '40um',
    lead = dict(
        start_straight = "20um",
        end_straight = "575um",
        end_jogged_extension = jogs   # was "start_jogged_extensions" (plural) — silently ignored
    ),
    # meander = dict(
    #     asymmetry = "800um"
    # ),
    total_length = "3.74575mm"
    # dropped the stray top-level "assymetry" — misspelled, unused, and duplicated meander.asymmetry
)

readout_1 = RouteMeander(design, "readout_1", options = meander_1_options)
gui.rebuild()
gui.autoscale()
